# TPE Optimizer (Optuna)

Joint-portfolio TPE search. See `docs/research.md` for methodology.

## Setup

In [ ]:
import importlib
import sys
from pathlib import Path

for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_candidate / "prediction_market_extensions").is_dir() and (
        _candidate / "backtests"
    ).is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

ensure_notebook_repo_context = importlib.import_module(
    "prediction_market_extensions.backtesting._notebook_support"
).ensure_notebook_repo_context
repo_root = ensure_notebook_repo_context()

## Configuration

In [ ]:
import os
from decimal import Decimal

from dotenv import load_dotenv

from prediction_market_extensions.backtesting._execution_config import ExecutionModelConfig
from prediction_market_extensions.backtesting._execution_config import StaticLatencyConfig
from prediction_market_extensions.backtesting._experiments import ParameterSearchExperiment
from prediction_market_extensions.backtesting._prediction_market_runner import MarketDataConfig
from prediction_market_extensions.backtesting._replay_specs import BookReplay
from prediction_market_extensions.backtesting.data_sources import Book, Polymarket, Telonex
from prediction_market_extensions.backtesting.optimizers import ParameterSearchConfig
from prediction_market_extensions.backtesting.optimizers import ParameterSearchWindow


load_dotenv(repo_root / ".env")
telonex_sources = ["local:/Volumes/LaCie/telonex_data"]
telonex_api_key = os.environ.get("TELONEX_API_KEY", "").strip()
if telonex_api_key:
    telonex_sources.append(f"api:{telonex_api_key}")

name = "notebook_tpe_optimizer"

data = MarketDataConfig(
    platform=Polymarket,
    data_type=Book,
    vendor=Telonex,
    sources=tuple(telonex_sources),
)

base_replays = (
    BookReplay(
        market_slug="will-openais-market-cap-be-between-750b-and-1t-at-market-close-on-ipo-day",
        token_index=0,
    ),
)

train_windows = (
    ParameterSearchWindow(
        name="train-2026-04",
        start_time="2026-04-05T00:00:00Z",
        end_time="2026-04-05T23:59:59Z",
    ),
)

holdout_windows = (
    ParameterSearchWindow(
        name="holdout-2026-04",
        start_time="2026-04-06T00:00:00Z",
        end_time="2026-04-06T23:59:59Z",
    ),
)

strategy_spec = {
    "strategy_path": "strategies:BookMicropriceImbalanceStrategy",
    "config_path": "strategies:BookMicropriceImbalanceConfig",
    "config": {
        "trade_size": Decimal("5"),
        "depth_levels": "__SEARCH__:depth_levels",
        "entry_imbalance": "__SEARCH__:entry_imbalance",
        "exit_imbalance": "__SEARCH__:exit_imbalance",
        "min_microprice_edge": "__SEARCH__:min_microprice_edge",
        "max_spread": "__SEARCH__:max_spread",
        "max_entry_price": "__SEARCH__:max_entry_price",
        "max_expected_slippage": "__SEARCH__:max_expected_slippage",
        "min_holding_updates": 0,
        "reentry_cooldown_updates": 0,
        "min_holding_seconds": "__SEARCH__:min_holding_seconds",
        "reentry_cooldown_seconds": "__SEARCH__:reentry_cooldown_seconds",
        "take_profit": "__SEARCH__:take_profit",
        "stop_loss": "__SEARCH__:stop_loss",
    },
}

parameter_space = {
    "depth_levels": {"type": "int", "low": 1, "high": 5},
    "entry_imbalance": {"type": "float", "low": 0.58, "high": 0.72},
    "exit_imbalance": {"type": "float", "low": 0.35, "high": 0.54},
    "min_microprice_edge": {"type": "float", "low": 0.0005, "high": 0.004},
    "max_spread": {"type": "float", "low": 0.015, "high": 0.06},
    "max_entry_price": {"type": "float", "low": 0.12, "high": 0.35},
    "max_expected_slippage": {"type": "float", "low": 0.003, "high": 0.015},
    "min_holding_seconds": {
        "type": "categorical",
        "choices": [300.0, 900.0, 1_800.0, 3_600.0],
    },
    "reentry_cooldown_seconds": {
        "type": "categorical",
        "choices": [600.0, 1_800.0, 3_600.0, 7_200.0],
    },
    "take_profit": {"type": "float", "low": 0.005, "high": 0.04},
    "stop_loss": {"type": "float", "low": 0.01, "high": 0.05},
}

execution = ExecutionModelConfig(
    queue_position=True,
    latency_model=StaticLatencyConfig(
        base_latency_ms=75.0,
        insert_latency_ms=10.0,
        update_latency_ms=5.0,
        cancel_latency_ms=5.0,
    ),
)

max_trials = 2
holdout_top_k = 1

parameter_search = ParameterSearchConfig(
    name=name,
    data=data,
    base_replays=base_replays,
    strategy_spec=strategy_spec,
    parameter_space=parameter_space,
    train_windows=train_windows,
    holdout_windows=holdout_windows,
    max_trials=max_trials,
    random_seed=7,
    holdout_top_k=holdout_top_k,
    initial_cash=100.0,
    probability_window=256,
    min_book_events=100,
    min_price_range=0.005,
    min_fills_per_window=1,
    execution=execution,
    sampler="tpe",
)

experiment = ParameterSearchExperiment(
    name=name,
    description="Microprice-imbalance TPE search on Telonex L2 book data",
    parameter_search=parameter_search,
)

print(
    f"Configured TPE microprice imbalance search: {max_trials} trials over "
    f"{len(base_replays)} market(s), {len(train_windows)} train window(s), "
    f"{len(holdout_windows)} holdout window(s)."
)

## Run optimizer

In [ ]:
from pprint import pprint

from prediction_market_extensions.backtesting._notebook_support import suppress_notebook_cell_output
from prediction_market_extensions.backtesting.optimizers import run_parameter_search

with suppress_notebook_cell_output():
    summary = run_parameter_search(parameter_search)

print("Selected params:")
pprint(dict(summary.selected_params))
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

## Leaderboard

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

_leaderboard_df = pd.read_csv(summary.leaderboard_csv_path)
_display_cols = [
    c
    for c in (
        "trial_id",
        "train_median_score",
        "holdout_median_score",
        "train_median_pnl",
        "holdout_median_pnl",
        "train_median_drawdown",
        "train_median_fills",
        "params_json",
    )
    if c in _leaderboard_df.columns
]
_top = _leaderboard_df[_display_cols].head(10).copy()

_fmt = {
    "train_median_score": "{:.4f}".format,
    "holdout_median_score": "{:.4f}".format,
    "train_median_pnl": "{:.4f}".format,
    "holdout_median_pnl": "{:.4f}".format,
    "train_median_drawdown": "{:.4f}".format,
    "train_median_fills": "{:.1f}".format,
}
_fmt = {k: v for k, v in _fmt.items() if k in _top.columns}

_styled = (
    _top.style.format(_fmt, na_rep="-")
    .hide(axis="index")
    .set_caption(f"Top candidates — {experiment.name} (TPE)")
)
display(_styled)

display(
    Markdown(
        "### Selected parameters\n\n"
        + "\n".join(f"- **{k}**: `{v}`" for k, v in dict(summary.selected_params).items())
    )
)

## Combined holdout chart

In [ ]:
import gc

from prediction_market_extensions.backtesting._notebook_support import (
    display_html_artifacts,
    find_updated_html_artifacts,
    snapshot_html_artifacts,
    suppress_notebook_cell_output,
)
from prediction_market_extensions.backtesting._prediction_market_backtest import MarketReportConfig
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)
from prediction_market_extensions.backtesting.prediction_market import finalize_market_results

selected_window = holdout_windows[0] if holdout_windows else train_windows[-1]

output_root = repo_root / "output"
html_snapshot = snapshot_html_artifacts(output_root)

plot_panels = (
    "total_equity",
    "equity",
    "market_pnl",
    "periodic_pnl",
    "yes_price",
    "allocation",
    "total_drawdown",
    "drawdown",
    "total_rolling_sharpe",
    "rolling_sharpe",
    "total_cash_equity",
    "cash_equity",
    "monthly_returns",
    "total_brier_advantage",
    "brier_advantage",
)

report = MarketReportConfig(
    count_key="book_events",
    count_label="Book Events",
    pnl_label="PnL (pUSD)",
    market_key="slug",
    summary_report=True,
    summary_report_path=f"output/{name}_holdout_joint_portfolio.html",
    summary_plot_panels=plot_panels,
)

backtest = build_parameter_search_window_backtest(
    config=parameter_search,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{name}:{selected_window.name}:selected-params",
    return_summary_series=True,
)

with suppress_notebook_cell_output():
    results = await backtest.run_async()
    finalize_market_results(
        name=backtest.name,
        results=results,
        report=report,
    )

updated_html = find_updated_html_artifacts(output_root, html_snapshot)
num_markets = len(results)
print(f"Replayed {num_markets} joint-portfolio market(s) on window '{selected_window.name}'.")
summary_html = repo_root / report.summary_report_path
display_html_artifacts([summary_html] if summary_html.exists() else [], repo_root=repo_root)

del results, backtest, updated_html, html_snapshot, summary_html
gc.collect()